In [2]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [3]:
df = pd.read_csv("../data/adult.csv", na_values="?")
df.dropna()
df = df.rename(columns={'salary K$': 'salary_k'})
df['salary_limit'] = df['salary'].str.extract(r'(\d+)').astype(int)
df['salary_op'] = df['salary'].str.extract(r'(<=|>)')
df['salary_k'] = pd.to_numeric(df['salary_k'], errors='coerce')
df = df.dropna(subset=['salary_k'])
salary_check = (
    ((df['salary_op'] == '<=') & (df['salary_k'] <= df['salary_limit'])) |
    ((df['salary_op'] == '>')  & (df['salary_k'] >  df['salary_limit']))
)
df = df[salary_check]
df

,Unnamed: 0,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,native-country,salary,salary_k,salary_limit,salary_op
0,0,39,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,40,United-States,<=50K,39,50,<=
1,1,50,Self-emp-not-inc,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,13,United-States,<=50K,35,50,<=
2,2,38,Private,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,40,United-States,<=50K,27,50,<=
3,3,53,Private,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,40,United-States,<=50K,43,50,<=
4,4,28,Private,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,40,Cuba,<=50K,25,50,<=
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,32556,27,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,38,United-States,<=50K,36,50,<=
32557,32557,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,40,United-States,>50K,173,50,>
32558,32558,58,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,40,United-States,<=50K,40,50,<=
32559,32559,22,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,20,United-States,<=50K,38,50,<=


# Task 1
Print the count of men and women in the dataset.

In [4]:
gendr_count = df.value_counts('sex')
print(f"Men: {gendr_count.get('Male', 0)}")
print(f"Women: {gendr_count.get('Female', 0)}")

Men: 21790
Women: 10771


# Task 2
Find the average age of men in dataset

In [5]:
avg_age_men = df.loc[df['sex'] == 'Male', 'age'].mean()
print(f"Average age of men: {avg_age_men:.0f}")

Average age of men: 39


# Task 3
Get the percentage of people from Poland (native-country)

In [8]:
percentage_poland = (df['native-country'].eq('Poland').mean()) * 100
print(f"Percentage people from Poland: {percentage_poland:.2f}%")

Percentage people from Poland: 0.18%


# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [11]:
greater_50 = df[df['salary'].str.startswith('>')]
lower_50 = df[df['salary'].str.startswith('<=')]
mean_age_1 = greater_50['age'].mean()
std_age_1 = greater_50['age'].std()
mean_age_2 = lower_50['age'].mean()
std_age_2 = lower_50['age'].std()
print(f"Mean age for salary greater than 50: {mean_age_1:.2f}, and std:{std_age_1:.2f}")
print(f"Mean age for salary lower than 50: {mean_age_2:.2f}, and std:{std_age_2:.2f}")

Mean age for salary greater than 50: 44.25, and std:10.52
Mean age for salary lower than 50: 36.78, and std:14.02


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [13]:
higher_education = {
    'Bachelors',
    'Prof-school',
    'Assoc-acdm',
    'Assoc-voc',
    'Masters',
    'Doctorate'
}

result = df[(df['salary'].str.startswith('>')) & (~df['education'].isin(higher_education))]
is_without_education = not result.empty
print(f"Does somebody with salary >50K and without education: {is_without_education}")

Does somebody with salary >50K and without education: True


# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [15]:
age_by_education = df.groupby('education')['age'].describe()
age_by_education

,count,mean,std,min,25%,50%,75%,max
education,,,,,,,,
10th,933.0,37.429796,16.720713,17.0,22.00,34.0,52.0,90.0
11th,1175.0,32.355745,15.545485,17.0,18.00,28.0,43.0,90.0
12th,433.0,32.000000,14.334625,17.0,19.00,28.0,41.0,79.0
1st-4th,168.0,46.142857,15.615625,19.0,33.00,46.0,57.0,90.0
5th-6th,333.0,42.885886,15.557285,17.0,29.00,42.0,54.0,84.0
7th-8th,646.0,48.445820,16.092350,17.0,34.25,50.0,61.0,90.0
9th,514.0,41.060311,15.946862,17.0,28.00,39.0,54.0,90.0
Assoc-acdm,1067.0,37.381443,11.095177,19.0,29.00,36.0,44.0,90.0
Assoc-voc,1382.0,38.553546,11.631300,19.0,30.00,37.0,46.0,84.0


# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [29]:
men = df[df['sex'] == 'Male']
married = men[men['marital-status'].str.startswith('Married')]
not_married = men[~men['marital-status'].str.startswith('Married')]
married_salary_dist = married['salary'].value_counts(normalize=True) * 100
not_married_salary_dist = not_married['salary'].value_counts(normalize=True) * 100
married_high = married_salary_dist.get('>50K', 0)
not_married_high = not_married_salary_dist.get('>50K', 0)

print(
    "Married men earn more on average (higher % with >50K)."
    if married_high > not_married_high
    else "Non-married men earn more on average."
)


Married men earn more on average (higher % with >50K).


# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [31]:
max_hours = df['hours-per-week'].max()
num_people = df['hours-per-week'].value_counts().loc[max_hours]
print(f"Max hours per week: {max_hours}")
print(f"Number of people: {num_people}")

Max hours per week: 99
Number of people: 85


# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [43]:
numeric_df = df.select_dtypes(include='number')
numeric_df = numeric_df.drop(columns=['salary_limit'], errors='ignore')
numeric_df = numeric_df.drop(columns=['Unnamed: 0'])
correlation_matrix = numeric_df.corr()

strong_corr = (
    correlation_matrix
    .stack()
    .reset_index()
    .rename(columns={0: 'correlation'})
)

strong_corr = strong_corr[
    (strong_corr['level_0'] != strong_corr['level_1']) &
    (strong_corr['correlation'].abs() >= 0.2)
]

strong_corr.sort_values(by='correlation', ascending=False)

for _, row in strong_corr.iterrows():
    print(
        f"{row['level_0']} ↔ {row['level_1']} : "
        f"correlation = {row['correlation']:.2f}"
    )


age ↔ salary_k : correlation = 0.20
salary_k ↔ age : correlation = 0.20
